# 1. Definição do problema

Este trabalho tem como objetivo desenvolver um modelo de Machine Learning para apoio ao diagnóstico médico, conforme proposto no Tech Challenge da Fase 1. A solução busca analisar dados de exames e identificar padrões que auxiliem na classificação de possíveis condições clínicas, contribuindo para a triagem inicial e suporte à decisão médica.

Para isso, será utilizado o dataset Breast Cancer Wisconsin, sugerido no enunciado, contendo características extraídas de exames relacionados ao diagnóstico de câncer de mama. O objetivo é classificar os casos como benignos ou malignos a partir das variáveis disponíveis, aplicando técnicas de análise exploratória, pré-processamento, modelagem e avaliação dos resultados.

## 2. Carregamento dos dados em um dataframe pandas

In [ ]:
import pandas as pd

df = pd.read_csv("../../datasets/dataset-wisconsin-breast-cancer.csv")
# dados.groupby("diagnosis").describe()

## 3. Inspeção inicial dos dados
- Ver shape, tipos das colunas, amostras dos dados.
- Checar valores ausentes, duplicados, outliers e distribuição das variáveis.
- Confirmar se a variável alvo está correta.


In [ ]:
# 1. Verifica o tamanho da tabela (linhas, colunas)
print("Formato da tabela:", df.shape) # (569 linhas, 33 colunas)

# 2. Verifica os tipos de dados e se há valores nulos explícitos
print("\nInformações das colunas:")
df.info()

In [ ]:
df.isnull().sum().sort_values(ascending=False)

In [ ]:
df.head()

A partir dessa análise inicial, verifica-se que o dataset apresenta boa consistência. Foi identificada apenas uma coluna adicional, “Unnamed”, composta exclusivamente por valores nulos, provavelmente gerados durante a exportação do arquivo CSV. Por não possuir relevância analítica, essa coluna será removida. Além disso, a coluna id também será excluída, pois não contribui para a modelagem ou para a análise dos dados.

## 4. Limpeza dos dados

In [ ]:
df = df.drop(columns=['Unnamed: 32',"id"])

Após esse procedimento, temos apenas valores não nulos conforme pode ser visto abaixo:

In [ ]:
df.isnull().sum().sort_values(ascending=False)

Verificando se existem registros duplicados no dataframe:

In [ ]:
df.duplicated().sum()

Não existem valores duplicados. Agora vamos analisar a distribuição da variável alvo `diagnosis`.

In [ ]:
df['diagnosis'].value_counts()

A variável alvo apresenta uma distribuição moderadamente desbalanceada: 357 casos benignos e 212 malignos. Isso corresponde a aproximadamente 62,7% de casos benignos e 37,3% de casos malignos. Embora o desbalanceamento não seja extremo, ele deve ser considerado na avaliação do modelo, especialmente porque falsos negativos para casos malignos são mais críticos no contexto médico.

Além disso, será realizada a conversao da variável target `diagnosis` para formato numérico. No CSV original, essa coluna contém valores M para maligno e B para benigno, e, portanto, será convertida para 1 e 0, respectivamente.

In [ ]:
df['diagnosis'] = df['diagnosis'].map({'M': 1, 'B': 0})
df.head()

Por fim, verificou-se que algumas colunas estão fora do padrão, apresentando um espaço em branco no nome. Assim, com o intuito de padronizá-las, o código abaixo foi escrito.

In [ ]:
df.columns = df.columns.str.lower().str.replace(' ', '_')

# 5. Analisando correlação das variáveis

Existe um desafio nesse dataset, que é a quantidade de colunas. Isso pode dificultar a análise. Talvez seja necessário reduzir o escopo ou filtrar o que é realmente relevante. Como ponto de partida, não será verificada a correlação entre todas as variáveis entre si, mas apenas destas com a variável target `diagnosis`.

Apesar da alta correlação de algumas variáveis com o diagnóstico, a correlação deve ser interpretada apenas como uma medida de associação linear. Ela não implica causalidade e não substitui a etapa de modelagem. Além disso, algumas variáveis são fortemente correlacionadas entre si, como radius, perimeter e area, o que pode indicar multicolinearidade.

In [ ]:
corr = df.corr()[['diagnosis']].sort_values(by='diagnosis', ascending=False)

import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(6,10))
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.title('Correlação com a variável alvo')
plt.show()

A análise de correlação com a variável alvo indica que atributos relacionados ao tamanho e à irregularidade do tumor apresentam maior associação com o diagnóstico. Em especial, variáveis como concave points, perimeter, radius e area, principalmente em suas versões `worst` e `mean`, apresentam correlações elevadas, sugerindo forte capacidade preditiva na distinção entre casos benignos e malignos. Esses resultados são coerentes do ponto de vista clínico, uma vez que tumores malignos tendem a apresentar maior tamanho e contornos mais irregulares, o que justifica a relevância dessas características no modelo.

Agora, fazendo uma análise descritiva mais explícita

In [ ]:
df.describe().T

In [ ]:
# média dos dados agrupada por diagnóstico (0 benigno e 1 maligno).
df.groupby("diagnosis").mean().T

In [ ]:
# mediana dos dados agrupada por diagnóstico (0 benigno e 1 maligno).
df.groupby("diagnosis").median().T

As estatísticas descritivas indicam diferenças relevantes entre os grupos benigno e maligno. Em média, os casos malignos apresentam maiores valores em variáveis associadas ao tamanho e ao contorno do tumor, como radius, perimeter, area, concavity e concave_points. Isso reforça os achados da análise de correlação e sugere que essas variáveis possuem potencial preditivo para a classificação.

Distribuição da variável alvo:

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(5,4))
sns.countplot(data=df, x="diagnosis")
plt.title("Distribuição da variável alvo")
plt.xlabel("Diagnóstico: 0 = Benigno, 1 = Maligno")
plt.ylabel("Quantidade")
plt.show()

Boxplots das principais variáveis

In [ ]:
top_features = [
    "radius_mean", "radius_worst",
    "perimeter_mean", "perimeter_worst",
    "area_mean", "area_worst",
    "concavity_mean", "concavity_worst",
    "concave_points_mean", "concave_points_worst"
]

for col in top_features:
    plt.figure(figsize=(6,4))
    sns.boxplot(data=df, x="diagnosis", y=col)
    plt.title(f"Distribuição de {col} por diagnóstico")
    plt.xlabel("Diagnóstico: 0 = Benigno, 1 = Maligno")
    plt.ylabel(col)
    plt.show()

Refazendo a análise acima, mas agora com a padronização das variáveis (apenas para visualização).

In [ ]:
from sklearn.preprocessing import StandardScaler

# Padronização usada apenas para visualização dos boxplots,
# não utilizada no treinamento dos modelos.
df_scaled_visualizacao  = df.copy()

scaler = StandardScaler()
df_scaled_visualizacao [top_features] = scaler.fit_transform(df_scaled[top_features])

df_long_scaled = df_scaled_visualizacao.melt(
    id_vars="diagnosis",
    value_vars=top_features,
    var_name="feature",
    value_name="valor_padronizado"
)

plt.figure(figsize=(14, 6))
sns.boxplot(
    data=df_long_scaled,
    x="feature",
    y="valor_padronizado",
    hue="diagnosis"
)

plt.xticks(rotation=45, ha="right")
plt.title("Boxplots padronizados das principais variáveis por diagnóstico")
plt.xlabel("Variável")
plt.ylabel("Valor padronizado")
plt.legend(title="Diagnóstico")
plt.tight_layout()
plt.show()

Os boxplots padronizados permitem comparar variáveis em diferentes escalas. Observa-se que várias características dos grupos mean e worst apresentam separação visual entre diagnósticos benignos e malignos. As variáveis do grupo worst tendem a apresentar maior separação entre as classes, especialmente aquelas relacionadas ao tamanho e à irregularidade do tumor.

### Análise dos outliers

Os boxplots indicam a presença de valores extremos em algumas variáveis, especialmente nas características relacionadas a área, concavidade e pontos côncavos. Esses valores aparecem como pontos fora dos limites dos boxplots.

No entanto, por se tratar de um dataset médico, esses outliers não devem ser removidos automaticamente. Valores extremos podem representar casos clinicamente relevantes, como tumores com maior tamanho ou maior irregularidade. Além disso, não foram identificados indícios de valores impossíveis ou erros evidentes de preenchimento.

Dessa forma, optou-se por manter os outliers no conjunto de dados. Para reduzir o impacto de diferenças de escala entre as variáveis, foi utilizado escalonamento nas etapas de modelagem dos algoritmos sensíveis à escala, como Regressão Logística e KNN. Modelos baseados em árvores, como Random Forest, tendem a ser menos sensíveis a outliers.

## 6. Treinamento do modelo: Separar treino e teste

Embora algumas variáveis apresentem alta correlação entre si, optou-se inicialmente por manter todas as variáveis preditoras válidas no treinamento dos modelos. Essa escolha evita a remoção prematura de informações potencialmente úteis.

In [ ]:
X = df.drop(columns="diagnosis")
y = df["diagnosis"]

In [ ]:
X.shape, y.shape

Foram separadas as variáveis preditoras, representadas por X, e a variável alvo, representada por y. Todas as variáveis numéricas válidas foram mantidas inicialmente no treinamento, evitando a remoção prematura de atributos potencialmente úteis.

Separação entre treino e teste — 80/20

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [ ]:
X_train.shape, X_test.shape

In [ ]:
y_train.value_counts(normalize=True)

In [ ]:
y_test.value_counts(normalize=True)

Os dados foram divididos em 80% para treino e 20% para teste. A divisão foi feita de forma estratificada, preservando proporções semelhantes de casos benignos e malignos nos dois conjuntos. O conjunto de teste será utilizado apenas na avaliação final do modelo escolhido.

Como foi adotada uma divisão principal em treino e teste, a etapa de validação dos modelos será realizada por validação cruzada dentro do conjunto de treino. Dessa forma, o conjunto de teste permanece isolado até a avaliação final.

## 7. Definição dos modelos

Foram selecionados três modelos de classificação: Regressão Logística, KNN e Random Forest.

A `Regressão Logística` foi utilizada como modelo de referência por ser simples, interpretável e adequada para problemas de classificação binária. O `KNN` foi incluído por ser um modelo baseado em proximidade entre observações, exigindo padronização das variáveis. A `Random Forest` foi utilizada por sua capacidade de capturar relações não lineares e por permitir, em análises complementares, a avaliação da importância relativa das variáveis.

Para a Regressão Logística e para o KNN, foi usado o `StandardScaler()`, visto que esses modelos são sensíveis à diferença de escala entre as variáveis. A variável `area_mean` pode ter valores muito maiores do que `concavity_mean`. Sem padronização, modelos baseados em distância ou coeficientes podem dar peso excessivo às variáveis de maior escala.

### Escolha do melhor K para o KNN por validação cruzada

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_validate, StratifiedKFold
import pandas as pd
import matplotlib.pyplot as plt

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

k_values = range(1, 31) # possíveis valores para K

knn_results = []

for k in k_values:
    knn_pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=k))
    ])

    scores = cross_validate(
        knn_pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring={
            "accuracy": "accuracy",
            "precision": "precision",
            "recall": "recall",
            "f1": "f1"
        }
    )

    knn_results.append({
        "k": k,
        "accuracy_mean": scores["test_accuracy"].mean(),
        "precision_mean": scores["test_precision"].mean(),
        "recall_mean": scores["test_recall"].mean(),
        "f1_mean": scores["test_f1"].mean()
    })

knn_results_df = pd.DataFrame(knn_results)

knn_results_df.sort_values(by="recall_mean", ascending=False).head(10) # ordenando pelo recall que é o mais importante no caso em questão.

### Gráfico de apoio à escolha do melhor K

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(knn_results_df["k"], knn_results_df["recall_mean"], marker="o", label="Recall")
plt.plot(knn_results_df["k"], knn_results_df["f1_mean"], marker="o", label="F1-score")

plt.title("Desempenho do KNN por valor de K")
plt.xlabel("Número de vizinhos (K)")
plt.ylabel("Métrica média na validação cruzada")
plt.xticks(list(k_values))
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Seleção do modelo priorizando recall da classe maligna
best_k_recall = knn_results_df.sort_values(
    by=["recall_mean", "f1_mean"],
    ascending=False
).iloc[0]

best_k_recall

A análise do desempenho do KNN para diferentes valores de `k` indicou que `k = 3` apresentou o melhor desempenho entre os valores testados, com maior F1-score e um dos maiores valores de recall. Como o problema envolve apoio ao diagnóstico de câncer de mama, o recall da classe maligna foi considerado uma métrica prioritária, pois está relacionado à capacidade do modelo de identificar corretamente os casos malignos.

Dessa forma, o valor `k = 3` foi selecionado para o modelo KNN, substituindo o valor padrão `k = 5`.

### Execução dos modelos:

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

models = {

    "Regressão Logística": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        ))
    ]),

    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=3))
    ]),

    "Random Forest": Pipeline([
        ("model", RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=42
        ))
    ])
}

### Justificativa dos hiperparâmetros

Os modelos foram configurados com ajustes simples e justificáveis, sem realização de uma busca exaustiva de hiperparâmetros. O objetivo foi construir uma comparação inicial consistente entre diferentes algoritmos de classificação.

Para a **Regressão Logística** e o **KNN**, foi utilizado `StandardScaler`, pois ambos são sensíveis à escala das variáveis. Como o dataset possui atributos em escalas diferentes, a padronização evita que variáveis com valores numéricos maiores tenham influência desproporcional no treinamento.

Na **Regressão Logística**, foi definido `max_iter=1000` para reduzir o risco de problemas de convergência durante o treinamento. Também foi utilizado `class_weight="balanced"` para compensar a diferença entre as classes, já que há mais casos benignos do que malignos. Essa decisão é relevante porque, no contexto médico, falsos negativos para a classe maligna são erros especialmente críticos.

No **KNN**, foi mantido `n_neighbors=3`, que corresponde ao valor encontrado anteriormente. Essa configuração foi usada como ponto de partida para comparação com os demais modelos.

Na **Random Forest**, foi utilizado `n_estimators=300`, aumentando o número de árvores em relação ao valor padrão. Esse ajuste tende a tornar as previsões mais estáveis e reduzir a variância do modelo. Como o dataset é pequeno, o aumento no custo computacional é baixo. Também foi utilizado `class_weight="balanced"` para reduzir o favorecimento da classe majoritária.

Por fim, o parâmetro `random_state=42` foi definido nos modelos que possuem componentes aleatórios, garantindo maior reprodutibilidade dos resultados.

Esses hiperparâmetros não representam necessariamente a configuração ótima dos modelos. Eles foram utilizados como baseline ajustado, priorizando reprodutibilidade, estabilidade e tratamento básico do desbalanceamento entre classes.

In [ ]:
from sklearn.model_selection import cross_validate, StratifiedKFold

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1"
}

cv_results = []

for name, model in models.items():
    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=False
    )

    cv_results.append({
        "modelo": name,
        "accuracy_mean": scores["test_accuracy"].mean(),
        "precision_mean": scores["test_precision"].mean(),
        "recall_mean": scores["test_recall"].mean(),
        "f1_mean": scores["test_f1"].mean()
    })

cv_results_df = pd.DataFrame(cv_results)
cv_results_df.sort_values(by="recall_mean", ascending=False)

A validação cruzada foi realizada apenas sobre o conjunto de treino. Essa estratégia permite comparar os modelos sem utilizar o conjunto de teste durante a escolha do melhor modelo.

A métrica de maior interesse neste problema é o recall da classe maligna, pois falsos negativos representam casos malignos classificados incorretamente como benignos. Em contexto médico, esse tipo de erro pode ter impacto mais grave do que um falso positivo.

## 8. Escolha do melhor modelo

A partir da validação cruzada realizada no conjunto de treino, a Regressão Logística foi selecionada como modelo final por apresentar o maior recall médio para a classe maligna na validação cruzada.

Embora o KNN tenha obtido maior accuracy, precision e F1-score, a prioridade do problema foi reduzir falsos negativos, já que classificar um caso maligno como benigno representa o erro mais crítico no contexto de apoio ao diagnóstico.

Dessa forma, a Regressão Logística foi selecionada como o melhor modelo para avaliação final no conjunto de teste, por apresentar o maior recall e o melhor equilíbrio geral entre as métricas.

Treinamento final com os 80% dos dados do dataset

In [ ]:
best_model_name = cv_results_df.sort_values(
    by="recall_mean",
    ascending=False
).iloc[0]["modelo"]

best_model_name

In [ ]:
best_model = models[best_model_name]

Treinamento final com os 80% de treino. Após a comparação dos modelos por validação cruzada, o modelo selecionado foi treinado novamente utilizando todo o conjunto de treino.

In [ ]:
best_model.fit(X_train, y_train)

## 9. Avaliação final no conjunto de teste

In [ ]:
y_pred = best_model.predict(X_test)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    y_test,
    y_pred,
    target_names=["Benigno", "Maligno"]
))

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

test_metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred, pos_label=1),
    "recall": recall_score(y_test, y_pred, pos_label=1),
    "f1_score": f1_score(y_test, y_pred, pos_label=1)
}

test_metrics

### Avaliação final no conjunto de teste (Interpretação)

A avaliação final foi realizada no conjunto de teste, que não foi utilizado durante o treinamento nem durante a seleção do modelo. Essa separação é importante para verificar a capacidade de generalização do modelo em dados ainda não vistos.

Foram analisadas as métricas **accuracy**, **precision**, **recall** e **F1-score**. Embora a **accuracy** indique o percentual geral de acertos, ela não deve ser interpretada isoladamente neste problema. Como se trata de um contexto médico, a métrica mais relevante é o **recall da classe maligna**, pois mede a capacidade do modelo de identificar corretamente os casos malignos.

No conjunto de teste, havia:

- **72 casos benignos**
- **42 casos malignos**
- **114 registros no total**

A Regressão Logística apresentou accuracy global de 0,9737. Para a classe maligna, os valores foram precision de 0,9756, recall de 0,9524 e F1-score de 0,9639.

| Métrica | Resultado |
|---|---:|
| Accuracy | 0,9737 |
| Precision | 0,9756 |
| Recall | 0,9524 |
| F1-score | 0,9639 |

Os resultados são coerentes com o desempenho observado na validação cruzada. A Regressão Logística manteve alto desempenho no conjunto de teste, indicando boa capacidade de generalização para dados não utilizados durante o treinamento.

O **recall da classe maligna** foi de aproximadamente **95,24%**, o que significa que o modelo identificou corretamente cerca de **95% dos casos malignos** presentes no conjunto de teste. Considerando que havia 42 casos malignos, esse resultado indica que o modelo classificou corretamente aproximadamente **40 casos malignos** e deixou de identificar cerca de **2 casos malignos**, classificando-os incorretamente como benignos.

A **precision da classe maligna** foi de aproximadamente **97,56%**, indicando que, quando o modelo classificou um caso como maligno, a taxa de acerto foi elevada.

### Interpretação crítica

Apesar do desempenho geral elevado, os **falsos negativos** continuam sendo o principal ponto de atenção. Em um problema de apoio ao diagnóstico de câncer de mama, falsos negativos representam casos malignos classificados incorretamente como benignos, o que pode atrasar a investigação clínica e o tratamento adequado.

Portanto, embora o modelo apresente bom desempenho, ele não deve ser utilizado como ferramenta autônoma de diagnóstico. Seu uso mais adequado seria como instrumento de **apoio à triagem e à decisão médica**, auxiliando na priorização de casos e na análise inicial dos dados. A decisão final deve permanecer sob responsabilidade da equipe médica.

### Matriz de confusão

A matriz de confusão permite observar diretamente os erros do modelo. O principal ponto de atenção são os falsos negativos, isto é, casos malignos classificados como benignos.

In [ ]:
# Matriz de confusão
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Benigno", "Maligno"]
)

disp.plot()
plt.title("Matriz de confusão - Modelo final")
plt.show()

## 10. Discussão crítica

Os resultados indicam que modelos de Machine Learning podem auxiliar na classificação inicial de tumores benignos e malignos a partir de dados estruturados. No entanto, o modelo não deve ser interpretado como uma ferramenta autônoma de diagnóstico.

Em uma aplicação real, o sistema poderia funcionar como apoio à triagem, destacando casos com maior probabilidade de malignidade para análise prioritária da equipe médica. Ainda assim, a decisão final deve permanecer sob responsabilidade do profissional de saúde.

Também existem limitações importantes. O dataset possui tamanho limitado e foi obtido em contexto controlado. Antes de qualquer uso prático, seria necessário validar o modelo com bases externas, dados de diferentes populações, dados coletados em condições reais e acompanhamento clínico adequado. Além disso, aspectos éticos, regulatórios, privacidade dos dados e impacto de falsos negativos precisam ser avaliados.

Uma possível melhoria técnica seria ajustar o limiar de decisão da Regressão Logística. Como o objetivo é reduzir falsos negativos, poderia ser utilizado um threshold menor que 0,5 para aumentar a sensibilidade do modelo à classe maligna, ainda que isso aumente o número de falsos positivos.

## 11. Conclusão

Neste trabalho, foi desenvolvido um fluxo de Machine Learning para classificação de tumores de mama como benignos ou malignos. O processo incluiu carregamento dos dados, limpeza, análise exploratória, análise de correlação, separação entre treino e teste, validação cruzada no conjunto de treino, treinamento de diferentes modelos de classificação e avaliação final no conjunto de teste.

A métrica de recall da classe maligna foi priorizada por sua relevância no contexto médico. Os resultados mostram que os modelos avaliados conseguem identificar padrões relevantes nos dados, principalmente em variáveis associadas ao tamanho e à irregularidade do tumor. Apesar do bom desempenho, o modelo deve ser entendido como ferramenta de apoio à decisão, e não como substituto da avaliação médica.